# ZelusBench: Attention Simple

**Baseline: all difficulty knobs at easy.**

Shallow chains, linear structure, no distractors, no transforms, 2D.
Models should score near-perfectly here.

- 15 scenarios, 45 queries

In [ ]:
import kaggle_benchmarks as kbench
import numpy as np
import re
import time

from zelusbench.scenarios.config import (
    ScenarioConfig, DAGStructure, QueryType,
    DistractorLevel, TransformDensity,
)
from zelusbench.scenarios.generator import ScenarioGenerator
from zelusbench.evaluation.parser import parse_model_response
from zelusbench.evaluation.scorer import score_query


def score_response(response_text, scenario):
    query_dicts = [q.to_dict() for q in scenario.queries]
    parts = re.split(r'\\[Query\\s+q_\\d+\\]', response_text)
    if len(parts) > 1:
        parts = parts[1:]
    if len(parts) != len(query_dicts):
        parts = [response_text] * len(query_dicts)
    return [score_query(qd, parse_model_response(rp, qd))
            for qd, rp in zip(query_dicts, parts)]


SEEDS = 15
print(f"ZelusBench Attention Simple")
print(f"Seeds: {SEEDS} | Total: {SEEDS} scenarios")

In [ ]:
@kbench.task(name="zelusbench_attn_simple")
def zelusbench_attn_simple(llm) -> tuple[float, float]:
    """Combined easy: all knobs at minimum difficulty.
    Returns (mean_accuracy, std_dev)."""

    _start = time.time()
    print(f"Running {SEEDS} easy scenarios...")
    print("=" * 60)

    all_scores = []

    for seed in range(SEEDS):
        cfg = ScenarioConfig(
            dim=2,
            min_chain_depth=2, max_chain_depth=3,
            dag_structure=DAGStructure.LINEAR,
            distractor_level=DistractorLevel.CLEAN,
            transform_density=TransformDensity.STATIC,
            query_types=[QueryType.POSITION],
            point_def_types=["cartesian_offset", "magnitude_polar"],
            num_queries=3, num_splits=3,
            seed=seed,
        )
        gen = ScenarioGenerator(cfg)
        scenario = gen.generate(f"simple_s{seed}")

        response = llm.prompt(scenario.prompt)
        scores = score_response(response, scenario)
        all_scores.extend(scores)

        avg = float(np.mean([s.score for s in scores]))
        tiers = [s.tier.name for s in scores]
        print(f"  [{seed+1}/{SEEDS}] avg={avg:.2f} tiers={tiers}")

    overall = float(np.mean([s.score for s in all_scores]))
    std = float(np.std([s.score for s in all_scores]))
    elapsed = time.time() - _start

    print(f"\nOverall: {overall:.3f} +/- {std:.3f}")
    print(f"Total queries: {len(all_scores)}")
    print(f"Time: {elapsed:.1f}s ({elapsed/60:.1f} min)")

    kbench.assertions.assert_true(
        overall >= 0,
        expectation=f"Simple: model should produce valid answers (got {overall:.3f})"
    )

    return overall, std


zelusbench_attn_simple.run(llm=kbench.llm)

In [ ]:
%choose zelusbench_attn_simple